# Download from WRDS (暂时不用)

In [ ]:
import wrds
import pandas as pd
import numpy as np
import sqlite3

def download_wrds_to_sqlite_robust(db_path='tail_risk_data.db', start_date='1990-01-01', end_date='2023-12-31'):
    print(f"=== Starting WRDS Micro Data Pipeline (Robust Version) ===")
    conn = sqlite3.connect(db_path)
    
    print("Connecting to WRDS... (This will take a while)")
    db = wrds.Connection()
    
    # ---------------------------------------------------------
    # 1. 提取 CRSP 月度量价与退市收益率 (Delisting Returns)
    # ---------------------------------------------------------
    print(">>> Fetching CRSP price & return data...")
    crsp_query = f"""
        SELECT a.permno, a.date, a.ret, a.prc, a.shrout, a.vol, b.exchcd, b.shrcd
        FROM crsp.msf AS a
        LEFT JOIN crsp.msenames AS b
        ON a.permno = b.permno AND b.namedt <= a.date AND a.date <= b.nameendt
        WHERE a.date BETWEEN '{start_date}' AND '{end_date}'
        AND b.shrcd IN (10, 11) AND b.exchcd IN (1, 2, 3)
    """
    crsp = db.raw_sql(crsp_query)
    crsp['date'] = pd.to_datetime(crsp['date']) + pd.offsets.MonthEnd(0)
    
    print(">>> Fetching CRSP delisting returns...")
    delist_query = f"""
        SELECT permno, dlstdt, dlret
        FROM crsp.msedelist
        WHERE dlstdt BETWEEN '{start_date}' AND '{end_date}'
    """
    delist = db.raw_sql(delist_query)
    delist['date'] = pd.to_datetime(delist['dlstdt']) + pd.offsets.MonthEnd(0)
    
    # 合并正常收益率与退市收益率 (解决尾部崩盘缺失问题)
    crsp = pd.merge(crsp, delist[['permno', 'date', 'dlret']], on=['permno', 'date'], how='left')
    
    # 根据学术标准计算真实收益率：(1 + ret) * (1 + dlret) - 1
    crsp['ret'] = crsp['ret'].fillna(0)
    crsp['dlret'] = crsp['dlret'].fillna(0)
    crsp['true_ret'] = (1 + crsp['ret']) * (1 + crsp['dlret']) - 1
    # 恢复因 fillna(0) 导致的无效 0 值
    crsp['true_ret'] = np.where((crsp['ret'] == 0) & (crsp['dlret'] == 0) & crsp['prc'].isna(), np.nan, crsp['true_ret'])
    crsp.drop(columns=['ret', 'dlret'], inplace=True)
    crsp.rename(columns={'true_ret': 'ret'}, inplace=True)
    
    crsp['me'] = np.abs(crsp['prc']) * crsp['shrout']
    
    # ---------------------------------------------------------
    # 2. 提取 Compustat 财务报表并处理冗余
    # ---------------------------------------------------------
    print(">>> Fetching Compustat fundamental data...")
    comp_query = f"""
        SELECT gvkey, datadate, at, ceq, ni, revt, cogs, xsga, xint
        FROM comp.funda
        WHERE datadate BETWEEN '{start_date}' AND '{end_date}'
        AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C'
    """
    comp = db.raw_sql(comp_query)
    comp['datadate'] = pd.to_datetime(comp['datadate'])
    
    # 处理财年变更导致的重复记录，保留最新的
    comp = comp.sort_values(['gvkey', 'datadate']).drop_duplicates(subset=['gvkey', 'datadate'], keep='last')
    
    # ---------------------------------------------------------
    # 3. 提取 CCM 链接表并对齐
    # ---------------------------------------------------------
    print(">>> Fetching CCM link table...")
    ccm_query = """
        SELECT gvkey, lpermno AS permno, linkdt, linkenddt
        FROM crsp.ccmxpf_linktable
        WHERE linktype IN ('LU', 'LC') AND linkprim IN ('P', 'C')
    """
    ccm = db.raw_sql(ccm_query)
    ccm['linkdt'] = pd.to_datetime(ccm['linkdt'])
    ccm['linkenddt'] = pd.to_datetime(ccm['linkenddt']).fillna(pd.to_datetime('today'))
    
    comp = pd.merge(comp, ccm, on='gvkey', how='inner')
    comp = comp[(comp['datadate'] >= comp['linkdt']) & (comp['datadate'] <= comp['linkenddt'])]
    
    # ---------------------------------------------------------
    # 4. 完美合并：6个月滞后 + 11个月向下填充 (Forward Fill)
    # ---------------------------------------------------------
    print(">>> Merging Data (Rolling Forward Fill Method)...")
    comp['merge_date'] = comp['datadate'] + pd.DateOffset(months=6) + pd.offsets.MonthEnd(0)
    
    crsp['year_month'] = crsp['date'].dt.to_period('M')
    comp['year_month'] = comp['merge_date'].dt.to_period('M')
    
    # 先做 Left Merge，此时只有财报发布后第 6 个月的那一天有数据
    micro_df = pd.merge(crsp, comp, on=['permno', 'year_month'], how='left')
    micro_df = micro_df.sort_values(['permno', 'date']).reset_index(drop=True)
    
    # 核心修复！对齐每只股票，将财报数据向下填充最多 11 个月
    acct_cols = ['at', 'ceq', 'ni', 'revt', 'cogs', 'xsga', 'xint']
    micro_df[acct_cols] = micro_df.groupby('permno')[acct_cols].ffill(limit=11)
    
    # ---------------------------------------------------------
    # 5. 存入数据库
    # ---------------------------------------------------------
    micro_df.drop(columns=['year_month', 'merge_date', 'gvkey', 'datadate', 'linkdt', 'linkenddt'], errors='ignore', inplace=True)
    print(f"Saving Micro Data to SQLite... (Shape: {micro_df.shape})")
    
    micro_df.to_sql('micro_data', conn, if_exists='replace', index=False)
    
    cursor = conn.cursor()
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_micro_date ON micro_data (date);")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_micro_permno ON micro_data (permno);")
    
    conn.close()
    db.close()
    print("=== WRDS Pipeline Complete! ===")

# 运行升级版管道
# download_wrds_to_sqlite_robust()

=== Starting WRDS Micro Data Pipeline ===
Connecting to WRDS... (This will take a while)
Loading library list...
Done
Saving Micro Data to SQLite... (Shape: (2035551, 20))
=== WRDS Pipeline Complete! ===


# Download fred data  (暂时不用)

In [ ]:
import pandas as pd
from pandas_datareader import data as pdr
import sqlite3

def download_fred_to_sqlite(db_path='tail_risk_data.db', start_date='1990-01-01', end_date='2023-12-31'):
    print(f"=== Starting FRED Macro Data Pipeline ===")
    conn = sqlite3.connect(db_path)
    
    # 为了计算同比或滞后，多拉取一年的历史数据
    start = pd.to_datetime(start_date) - pd.DateOffset(years=1)
    
    # 修复后的 Tickers：Moody's Baa 企业债为 'BAA'
    tickers = ['T10Y3M', 'BAA', 'AAA', 'VIXCLS']
    print(f"Fetching Tickers: {tickers}")
    macro_raw = pdr.get_data_fred(tickers, start, end_date)
    
    # 转换为月度末数据
    macro_df = macro_raw.resample('ME').last().reset_index() # Pandas 2.0+ 使用 'ME', 老版本使用 'M'
    macro_df.rename(columns={'DATE': 'date'}, inplace=True)
    macro_df['date'] = macro_df['date'] + pd.offsets.MonthEnd(0)
    
    # 计算宏观状态因子
    macro_df['TERM'] = macro_df['T10Y3M']
    macro_df['DEF'] = macro_df['BAA'] - macro_df['AAA']
    macro_df['VIX'] = macro_df['VIXCLS']
    
    # 剔除冗余列并处理缺失值
    macro_df = macro_df[['date', 'TERM', 'DEF', 'VIX']].ffill().dropna()
    
    print(f"Saving Macro Data to SQLite... (Shape: {macro_df.shape})")
    macro_df.to_sql('macro_data', conn, if_exists='replace', index=False)
    
    # 建立索引
    cursor = conn.cursor()
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_macro_date ON macro_data (date);")
    
    conn.close()
    print("=== FRED Pipeline Complete! ===")

download_fred_to_sqlite()

=== Starting FRED Macro Data Pipeline ===
Fetching Tickers: ['T10Y3M', 'BAA', 'AAA', 'VIXCLS']
Saving Macro Data to SQLite... (Shape: (408, 4))
=== FRED Pipeline Complete! ===


# Download features from open asset pricing  (暂时不用)

In [1]:
import openassetpricing as oap
import sqlite3
import pandas as pd

print("=== Downloading OAP Dataset via Official Python API (v3) ===")

try:
    openap = oap.OpenAP()
    print("Successfully connected to Open Asset Pricing servers.")
except Exception as e:
    print(f"Connection failed: {e}")

# 核心修正：使用我们刚刚侦测出来的正确 API 接口
print("Downloading 200+ firm characteristics (This will take a few minutes, ~1.6GB)...")
try:
    # 调用 dl_all_signals，通常默认返回 pandas dataframe
    df_features = openap.dl_all_signals('pandas')
except Exception as e:
    print(f"Download error: {e}")

print(f"Data downloaded successfully! Shape: {df_features.shape}")


    
# # 将缺失值填充或者处理掉 (特征表里通常有大量的 NaN，这里我们先保留存入数据库)
# print("Saving features to SQLite database as 'oap_features'...")
# with sqlite3.connect('tail_risk_data.db') as conn:
#     df_features.to_sql('oap_features', conn, if_exists='replace', index=False)

# print("=== OAP Pipeline Complete! You now possess the academic Holy Grail. ===")


=== Downloading OAP Dataset via Official Python API (v3) ===
Successfully connected to Open Asset Pricing servers.
WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done

Data is downloaded: 72 mins
Data downloaded successfully! Shape: (5416424, 214)


In [5]:
# save to Parquet
df_features["year"] = pd.to_datetime(df_features["yyyymm"], format='%Y%m').dt.year
df_features.to_parquet('oap_features.parquet',partition_cols=["year"], index=False)

In [ ]:
# read the Parquet file on year 2020
oap_2020 = pd.read_parquet('./data/oap_features.parquet', engine='pyarrow', filters=[('year', '==', 2020)])
oap_2020.head(5)

,permno,yyyymm,AM,AOP,AbnormalAccruals,Accruals,AccrualsBM,Activism1,Activism2,AdExp,...,skew1,std_turn,tang,zerotrade12M,zerotrade1M,zerotrade6M,Price,Size,STreversal,year
0,10026,202001,0.297053,0.140903,-0.036085,0.029798,NaN,NaN,NaN,0.001650,...,-0.050505,NaN,0.710366,8.212433e-08,2.013817e-08,1.457501e-07,-5.111023,-8.051190,0.100016,2020
1,10026,202002,0.306326,0.140903,-0.036085,0.029798,NaN,NaN,NaN,0.001701,...,-0.031910,NaN,0.710366,8.174433e-08,1.756096e-08,1.399235e-07,-5.080286,-8.020452,0.030270,2020
2,10026,202003,0.446013,0.140903,0.005110,0.040243,NaN,NaN,NaN,0.002598,...,0.235420,NaN,0.755241,8.005990e-08,2.336111e-08,1.452774e-07,-4.795791,-7.734317,0.244031,2020
3,10026,202004,0.424841,0.140903,0.005110,0.040243,NaN,NaN,NaN,0.002475,...,-0.142723,NaN,0.755241,7.113406e-08,9.570691e-09,1.271560e-07,-4.844423,-7.782950,-0.049835,2020
4,10026,202005,0.419556,0.140903,0.005110,0.040243,NaN,NaN,NaN,0.002444,...,0.020169,NaN,0.755241,6.537433e-08,1.103157e-08,1.143823e-07,-4.856940,-7.795467,-0.012596,2020


In [10]:
# read the Parquet file on year 2020
oap_2024 = pd.read_parquet('oap_features.parquet', engine='pyarrow', filters=[('year', '==', 2024)])
# oap_2020.head(5)
oap_2024[oap_2024['permno']==14593]

,permno,yyyymm,AM,AOP,AbnormalAccruals,Accruals,AccrualsBM,Activism1,Activism2,AdExp,AgeIPO,AnalystRevision,AnalystValue,AnnouncementReturn,AssetGrowth,BM,BMdec,BPEBM,Beta,BetaFP,BetaLiquidityPS,BetaTailRisk,BidAskSpread,BookLeverage,BrandInvest,CBOperProf,CF,CPVolSpread,Cash,CashProd,ChAssetTurnover,ChEQ,ChForecastAccrual,ChInv,ChInvIA,ChNAnalyst,ChNNCOA,ChNWC,ChTax,ChangeInRecommendation,CitationsRD,CompEquIss,CompositeDebtIssuance,ConsRecomm,ConvDebt,CoskewACX,Coskewness,CredRatDG,CustomerMomentum,DebtIssuance,DelBreadth,DelCOA,DelCOL,DelDRC,DelEqu,DelFINL,DelLTI,DelNetFin,DivInit,DivOmit,DivSeason,DivYieldST,DolVol,DownRecomm,EBM,EP,EarnSupBig,EarningsConsistency,EarningsForecastDisparity,EarningsStreak,EarningsSurprise,EntMult,EquityDuration,ExchSwitch,ExclExp,FEPS,FR,FirmAge,FirmAgeMom,ForecastDispersion,Frontier,GP,Governance,GrAdExp,GrLTNOA,GrSaleToGrInv,GrSaleToGrOverhead,Herf,HerfAsset,HerfBE,High52,IO_ShortInterest,IdioVol3F,IdioVolAHT,Illiquidity,IndIPO,IndMom,IndRetBig,IntMom,IntanBM,IntanCFP,IntanEP,IntanSP,InvGrowth,InvestPPEInv,Investment,LRreversal,Leverage,MRreversal,MS,MaxRet,MeanRankRevGrowth,Mom12m,Mom12mOffSeason,Mom6m,Mom6mJunk,MomOffSeason,MomOffSeason06YrPlus,MomOffSeason11YrPlus,MomOffSeason16YrPlus,MomRev,MomSeason,MomSeason06YrPlus,MomSeason11YrPlus,MomSeason16YrPlus,MomSeasonShort,MomVol,NOA,NetDebtFinance,NetDebtPrice,NetEquityFinance,NetPayoutYield,NumEarnIncrease,OPLeverage,OScore,OperProf,OperProfRD,OptionVolume1,OptionVolume2,OrderBacklog,OrderBacklogChg,OrgCap,PS,PatentsRD,PayoutYield,PctAcc,PctTotAcc,PredictedFE,PriceDelayRsq,PriceDelaySlope,PriceDelayTstat,ProbInformedTrading,RD,RDAbility,RDIPO,RDS,RDcap,REV6,RIO_Disp,RIO_MB,RIO_Turnover,RIO_Volatility,RIVolSpread,RealizedVol,Recomm_ShortInterest,ResidualMomentum,ReturnSkew,ReturnSkew3F,RevenueSurprise,RoE,SP,ShareIss1Y,ShareIss5Y,ShareRepurchase,ShareVol,ShortInterest,SmileSlope,Spinoff,SurpriseRD,Tax,TotalAccruals,TrendFactor,UpRecomm,VarCF,VolMkt,VolSD,VolumeTrend,XFIN,betaVIX,cfp,dCPVolSpread,dNoa,dVolCall,dVolPut,fgr5yrLag,grcapx,grcapx3y,hire,iomom_cust,iomom_supp,realestate,retConglomerate,roaq,sfe,sinAlgo,skew1,std_turn,tang,zerotrade12M,zerotrade1M,zerotrade6M,Price,Size,STreversal,year
11280,14593,202401,0.123883,NaN,0.039193,0.028740,NaN,NaN,NaN,NaN,NaN,1.003044,NaN,-0.009730,-0.004994,-3.772375,0.025024,-0.029920,0.852613,1.140004,0.083023,0.528579,0.002812,-6.848282,NaN,0.443503,0.038105,NaN,0.174583,-51.646400,NaN,-0.803170,1.0,0.004644,0.434971,NaN,0.027333,0.022068,0.000302,-0.75,NaN,2.330307,-0.135604,NaN,0.0,-0.065314,0.242924,0.0,NaN,-1.0,0.000,-0.042356,-0.064991,0.000853,0.035291,0.011487,0.020098,-0.017253,0.0,0.0,1.0,1.0,-12.241311,-1.0,-0.012125,0.032495,NaN,0.000157,-1.065922,0.000398,0.171097,-22.879096,-18.557155,0.0,0.01,6.59,NaN,-518.0,NaN,-0.025797,-2.531915,0.508801,NaN,NaN,0.063585,0.302024,-0.012262,-0.437346,-0.419787,-0.703442,0.930796,NaN,-0.008948,-0.009201,9.227538e-13,0.0,0.025249,NaN,0.497235,-2.839446,-0.572902,-0.595603,-3.003726,0.268010,-0.009772,-0.956379,0.008863,0.106088,0.046796,5.0,-0.032571,1697.200000,0.341622,0.029438,-0.017462,NaN,-0.038517,-0.013418,-0.030745,-0.046847,NaN,-0.050788,0.071526,0.052082,0.008915,0.023217,6.0,-0.319300,0.000350,NaN,0.313932,0.035951,2.0,0.754606,0.0,2.470911,0.437664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.035951,0.223921,-0.961514,NaN,0.005907,0.133022,0.134839,NaN,0.009219,NaN,0.0,-17217.50980,NaN,0.003541,NaN,1.0,NaN,NaN,NaN,-0.015161,NaN,-0.445704,-0.235788,-0.251334,-0.179830,1.969589,0.138483,0.027162,0.190762,1.0,NaN,-0.006558,NaN,0.0,1.0,0.483409,-0.000823,0.047788,0.0,-0.000235,-0.075685,-409.638695,-0.008463,0.313501,0.001255,0.042898,NaN,0.006054,NaN,NaN,-8.73,-0.465043,-1.111980,-0.062893,0.052311,-1.087367,NaN,NaN,0.068518,NaN,NaN,NaN,NaN,0.457494,9.727115e-08,3.232918e-08,2.034432e-07,-5.217107,-14.861946,0.042227,2024
11281,14593,202402,0.126385,NaN,0.039193,0.028740,NaN,NaN,NaN,NaN,NaN,0.995448,NaN,-0.034968,-0.004994,-3.772375,0.025024,-0.030532,0.843

这绝对是一座量化资产定价的“特征金矿”！你给出的这个长串，正是大名鼎鼎的 Open Asset Pricing (OAP) 或者是 Green, Hand, and Zhang (2017) 论文中整理的 **200+ 个经典异象（Anomalies）和预测因子（Predictors）**的完整集合。

要把这几百个变量一个一个解释完需要写一本书，但我已经为你把它们进行了**模块化解构**。特别是针对你想要预测极端左尾（如 $ES_{5\%}$）和条件概率密度的研究目标，我为你优先梳理了那些能够直接“撑大方差”或“拉扯左偏度”的核心特征组。

---

### 🔑 0. 底层主键 (Identifiers)
* **`permno`**: 股票的唯一识别码（CRSP 体系）。
* **`yyyymm`**: 年月（横截面的时间锚点）。

---

### 🚨 1. 尾部引爆器：风险、摩擦与微观结构 (Risk & Frictions)
这一组是你构建混合密度网络（MDN）时，**绝对不可或缺的特征**。它们平时可能不起眼，但在极端市场环境下，它们是触发崩盘、导致分布呈现“极厚左尾”的罪魁祸首。

* **波动率与极值类（撑大 $\sigma$）**：
    * **`IdioVol3F` / `IdioVolAHT`**: 特质波动率（剥离了市场因子的残差波动率）。历史波动越大的股票，未来的不确定性极高。
    * **`MaxRet`**: 过去一个月的最大单日收益率。这是经典的“彩票偏好”代理变量。散户越喜欢炒作的彩票股，未来的暴跌风险（负收益）越大。
    * **`RealizedVol`**: 已实现总波动率。
    * **`ReturnSkew` / `ReturnSkew3F`**: 收益率偏度。直接衡量历史分布是不对称的左偏还是右偏。
    * **`BetaTailRisk`**: 尾部风险贝塔！这个极其关键，衡量股票对宏观尾部风险的敏感度。
* **流动性枯竭类（触发左侧崩盘 $\pi$）**：
    * **`Illiquidity`**: Amihud 非流动性指标（绝对收益/交易额）。数值越大，一旦遭遇抛售，价格跌幅越深。
    * **`BidAskSpread`**: 买卖价差。微观结构摩擦的直接体现。
    * **`zerotrade1M` / `zerotrade6M`**: 过去1个月/6个月的零交易天数。流动性极差的标志。
    * **`PriceDelay` (相关指标)**: 价格对大盘信息的反应延迟程度。

---

### 📈 2. 趋势与均值回归：动量与反转 (Momentum & Reversal)
神经网络极其依赖这些指标来锚定股票收益率分布的“中心均值（$\mu$）”。

* **中期动量（强者恒强）**：
    * **`Mom12m` / `Mom6m`**: 经典的 12 个月和 6 个月动量（通常剔除最近1个月）。
    * **`IndMom`**: 行业动量。
    * **`ResidualMomentum`**: 剥离了系统风险后的残差动量，信号比纯动量更纯粹。
* **短期/长期反转（均值回归）**：
    * **`STreversal`**: 短期反转（过去1个月收益）。上个月刚暴跌的股票，下个月方差极大（可能反弹，也可能退市）。
    * **`LRreversal`**: 长期反转（过去 3-5 年的收益）。
* **季节性效应**：
    * **`MomSeason` / `MomOffSeason`**: 捕捉财报季或特定月份的季节性收益规律。

---

### 💼 3. 财务雷区：盈余质量与盈利能力 (Profitability & Accruals)
在基本面层面，这些指标能前瞻性地预测公司是否会“暴雷”。

* **盈余操纵预警（造假基因）**：
    * **`Accruals` / `AbnormalAccruals` / `PctAcc`**: 应计利润。账面上赚钱但没有现金流入，这是公司做假账或未来业绩暴雷的最强预测器。
* **盈利能力**：
    * **`RoE`** (净资产收益率), **`OperProf`** (营业利润率): 核心造血能力。
    * **`EarningsSurprise` / `SUE`**: 季报盈利超预期的幅度。
* **破产预警分数**：
    * **`OScore`**: Ohlson O-Score，经典的财务破产预测模型得分。分数越高，违约和左尾崩盘的概率越大。

---

### 🏛️ 4. 估值锚点：价值指标 (Value)
* **`BM` / `BMdec`**: 账面市值比。区分价值股和成长股。
* **`EP`** (盈利市值比), **`CP`** (现金流市值比), **`SP`** (销售额市值比)。
* **`IntanBM` / `IntanEP`**: 调整了无形资产（Intangibles）后的估值指标，对现代科技企业更有效。

---

### 🏭 5. 管理层行为：投资与融资 (Investment & Financing)
* **盲目扩张（通常是负向信号）**：
    * **`AssetGrowth`**: 总资产增长率。资产扩张过快的公司，未来收益往往拉胯。
    * **`InvestPPE` / `InvGrowth`**: 固定资产投资。
* **吸血行为（极其负面）**：
    * **`ShareIss1Y` / `ShareIss5Y`**: 股票增发。频繁在市场上抽血的公司，是对老股东权益的稀释。
    * **`DebtIssuance` / `NetDebtFinance`**: 发债规模。
* **回报股东（正面信号）**：
    * **`ShareRepurchase`**: 股份回购。
    * **`DivYield`**: 股息率。

---

### 💡 词缀“作弊”指南 (Naming Conventions)
当你在这 200 多个变量中迷失时，掌握这些词缀就能瞬间猜出含义：
* **`Ch` / `Del` / `d`** (如 `ChInv`, `DelCOA`, `dNoa`): Change / Delta，代表该指标的**变化量**。
* **`Gr`** (如 `GrSaleTo`, `grcapx`): Growth，代表该指标的**增长率**。
* **`Vol` / `Var` / `STD`** (如 `VolMkt`, `VarCF`): Volatility / Variance，代表**波动率或方差**。
* **`Ind`** (如 `IndMom`): Industry，代表该指标是经过**行业调整**或行业级别的。
* **`RD`** (如 `CitationsRD`, `PatentsRD`): Research & Development，研发相关的科技属性。

面对这 200+ 维度的特征全家桶，你是打算依靠经济学直觉手动挑选出一组 30 个左右的核心特征包，还是想通过主成分分析（PCA）等降维算法把它们压缩成几个隐变量后再喂给模型？

# List the tables in the database to verify

In [35]:
import pandas as pd
import numpy as np
import sqlite3
import os
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行
wrds_path = '/Users/jianbinchen/NonSync/GitHub/Research/DataSet/WRDS.sqlite'
# set current working directory to the location of the database file
os.chdir('/Users/jianbinchen/NonSync/GitHub/Research/DCDE')

# # list the tables in the database to verify
# with sqlite3.connect('tail_risk_data.db') as conn:
#     cursor = conn.cursor()
#     cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
#     tables = cursor.fetchall()
#     print("Tables in the database:", tables)

# Read the Data from Amit Goyal (宏观因子)

几个关键决策说明：

**为什么 `e12` 取 +2个月而不是 +1个月？**
S&P 500的盈利数据是企业盈利的加权汇总。按SEC规定，大多数企业需在季末45天内提交10-Q，但允许长达90天。Goyal的`e12`是滚动12个月的数据，最后一个季度盈利通常要等到该季结束后的2个月左右才能汇总到位。取 +2个月是保守估计，严格一点可以取 +3个月。

**`b/m` 为什么只需 +1季度，而不是学术界常说的 +6个月？**
关键在于这里的 `b/m` 是 **市场整体** 的账面市值比，账面价值来自 Fed Flow of Funds Z.1 报告（季度发布，季末约2.5个月后可得），而非 Fama-French 用个股年报数据构造的截面 `b/m`。后者才需要 +6个月（对应12月财年报告到次年6月才对外匹配）。

**三类"全样本偏差"的严重程度不同：**
- `cay` / `pce`：Lettau-Ludvigson 的协整关系需要全样本估计，bias **极严重**，要做 OOS 检验必须重算
- `ogap`：HP 滤波天然使用两端数据，边缘期的产出缺口估计会随未来数据修正
- `sntm` / `fbm`：PLS 权重在全样本上优化，OOS 实际可得的版本预测力会大幅下降
- `tchi` / `shtint`：程度相对轻一些，但 Goyal 仍明确标注需重估


# 选择的宏观因子
'''
* **12 tbl:** 短期国库券利率 (T-bill)。
* **19 tms (Term Spread):** 期限利差 (lty - tbl)。长短期利率倒挂是衰退的最强前兆。
* **20 dfy (Default Yield Spread):** 违约收益率差 (BAA - AAA)。衡量市场对企业破产的恐慌度。
* **21 dfr (Default Return Spread):** 违约回报率差 (corpr - ltr)。衡量企业债相对于长期国债的风险溢价。
* **22 infl:** 通货膨胀率。
* **25 svar ($\sigma^2$):** 历史股票方差（经典变量）。
* **30 vp, 方差溢价 (Variance Premium)
* **31 impvar, 隐含波动率
* **32 vrp:** 方差风险溢价。期权市场定价和实际波动的差值，是预测崩盘的神器。
* **35 skew & 43 skvw:** 偏度 (Skewness) 和股票平均偏度。捕捉市场收益分布的非对称性。
* **44 tail (X-sect tail risk):** 横截面尾部风险！这简直是你 $ES_{5\%}$ 的宏观对标物。
* **49 rdsp (Stock return dispersion):** 股票收益率分散度。横截面上股票各自为战的程度。
* **50 rsvix:** 缩放后的风险中性 VIX 指数。
* **55 avgcor:** 股票收益率的平均相关性。危机时相关性通常会飙升（“所有东西都在跌”）。
* **14 d/p (Dividend Price Ratio):** 股息价格比 (d12/price)。
* **15 d/y (Dividend Yield):** 股息率 (d12 / 上期price)。
* **16 e/p (Earnings Price Ratio):** 盈利价格比 (e12/price)。
* **17 d/e (Dividend Payout Ratio):** 股息支付率 (d12/e12)。企业利润中有多少拿来分红。
* **18 b/m (Book-to-Market):** 宏观账面市值比（道琼斯工业平均指数的 B/M）。
* **23 eqis & 24 ntis:** 股权发行比例和净股权发行。管理层是“聪明钱”，疯狂发股票圈钱往往意味着大盘见顶。
* **7 AAA & 8 BAA:** 最高信用评级 (AAA) 和中等信用评级 (BAA) 的企业债收益率。
* **9 lty & 10 ltr:** 长期国债的收益率 (Yield) 和实际回报率 (Return)。
* **57 disag:** 分析师预测分歧度 (Analyst disagreement)。分歧越大，往往未来收益越差。
'''

In [36]:
import pandas as pd
import numpy as np

file_path='/Users/jianbinchen/NonSync/GitHub/Research/DCDE/data/gwz_Data2024.xlsx'
df_monthly = pd.read_excel(file_path, sheet_name='Monthly')
df_quarterly = pd.read_excel(file_path, sheet_name='Quarterly')

# 日期
df_monthly['yyyymm'] = df_monthly['yyyymm'].astype(str)
df_monthly['date'] = pd.to_datetime(df_monthly['yyyymm'], format='%Y%m') + pd.offsets.MonthEnd(0)
df_quarterly['yyyyq'] = df_quarterly['yyyyq'].astype(str)
df_quarterly['yyyyq'] = df_quarterly['yyyyq'].str[:4] + 'Q' + df_quarterly['yyyyq'].str[4:]
# 1. 统一转换日期（使用 PeriodIndex，简洁且不容易出错）
df_quarterly['date'] = pd.PeriodIndex(df_quarterly['yyyyq'], freq='Q').to_timestamp(how='end')
# 2. 一步到位完成偏移（+4个月 + 月末校准）
# 这里的 DateOffset(months=4) 实际上已经涵盖了你之前逻辑中的 +3个月滞后 +1个月延后
df_quarterly['date'] = df_quarterly['date'] + pd.DateOffset(months=4) + pd.offsets.MonthEnd(0)
# 在所有计算完成后，加这一行即可抹掉时分秒
df_quarterly['date'] = pd.to_datetime(df_quarterly['date']).dt.normalize()
df_quarterly.tail(5)

/Users/jianbinchen/miniconda3/envs/research/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/Users/jianbinchen/miniconda3/envs/research/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,yyyyq,price,d12,e12,ret,retx,AAA,BAA,lty,ltr,corpr,tbl,Rfree,d/p,d/y,e/p,d/e,b/m,tms,dfy,dfr,infl,eqis,ntis,svar,cay,i/k,csp,pce,vp,impvar,vrp,govik,lzrt,skew,crdstd,ogap,wtexas,accrul,cfacc,sntm,ndrbl,skvw,tail,fbm,dtoy,dtoat,ygap,rdsp,rsvix,gpce,gip,tchi,house,avgcor,shtint,disag,date
611,2023Q4,4769.830000,70.303692,192.43,0.116447,0.111725,0.0474,0.0564,0.0402,0.056625,0.085022,0.0524,0.0133,0.014739,0.015391,0.040343,0.365347,0.194424,-0.0122,0.0090,0.028397,-0.003389,NaN,-0.017441,0.003816,0.007167,0.035341,NaN,0.029115,NaN,NaN,5.96,0.031671,-1.739012,NaN,33.9,-0.046715,-0.049200,NaN,NaN,NaN,0.022876,-0.000504,0.508539,0.341286,0.999455,0.999455,-3.249746,0.047711,NaN,NaN,NaN,0.792587,NaN,0.239503,-2.094452,3.351642,2024-04-30
612,2024Q1,5254.354401,70.824826,191.39,0.104158,0.100257,0.0501,0.0575,0.0421,-0.009565,-0.003992,0.0524,0.0131,0.013479,0.013897,0.036425,0.370055,0.195641,-0.0103,0.0074,0.005572,0.018211,NaN,-0.011172,0.003016,-0.004674,0.035367,NaN,0.026242,NaN,NaN,NaN,0.031867,-1.779969,NaN,14.5,-0.049169,0.059833,NaN,NaN,NaN,-0.001112,0.016063,0.499762,0.414574,1.000000,1.000000,-3.353736,0.033261,NaN,NaN,NaN,1.055849,NaN,0.166170,-2.151200,3.260390,2024-07-31
613,2024Q2,5460.482621,71.975800,195.93,0.045148,0.041568,0.0513,0.0582,0.0431,0.000949,-0.000879,0.0524,0.0131,0.013181,0.013638,0.035881,0.367355,0.199085,-0.0093,0.0069,-0.001828,0.005901,NaN,-0.014270,0.002742,-0.000927,0.035339,NaN,0.026872,NaN,NaN,NaN,0.031946,-1.793468,NaN,15.6,-0.043351,0.062332,NaN,NaN,NaN,-0.088776,0.027524,0.496907,0.371089,0.977884,0.977884,-3.369732,0.032274,NaN,NaN,NaN,1.055849,NaN,0.143822,-1.761255,2.717360,2024-10-31
614,2024Q3,5762.484883,73.400300,200.27,0.060385,0.056909,0.0468,0.0542,0.0372,0.047394,0.058374,0.0472,0.0131,0.012738,0.012995,0.034754,0.366507,0.183981,-0.0100,0.0074,0.010980,0.003584,NaN,-0.010283,0.006398,0.004238,0.035323,NaN,0.032010,NaN,NaN,NaN,0.032008,-1.915717,NaN,7.9,-0.051034,-0.077429,NaN,NaN,NaN,-0.003586,-0.035656,0.480061,0.413777,1.000000,1.000000,-3.395982,0.023585,NaN,NaN,NaN,1.055849,NaN,0.236572,-2.006363,2.263729,2025-01-31
615,2024Q4,5881.627648,74.832255,210.17,0.026559,0.023105,0.0520,0.0580,0.0439,-0.031382,-0.030355,0.0427,0.0118,0.012723,0.012405,0.035733,0.356056,0.183056,0.0012,0.0060,0.001026,0.000964,NaN,-0.004614,0.003970,0.004691,0.034675,NaN,0.037742,NaN,NaN,NaN,0.032110,-2.014421,NaN,0.0,-0.047211,0.061236,NaN,NaN,NaN,-0.033976,-0.077636,0.541364,NaN,0.945132,0.945132,-3.374636,0.029925,NaN,NaN,NaN,1.055849,NaN,0.195351,-2.135697,2.824905,2025-04-30


In [37]:
import pandas as pd
import numpy as np

file_path='/Users/jianbinchen/NonSync/GitHub/Research/DCDE/data/gwz_Data2024.xlsx'
df = pd.read_excel(file_path, sheet_name='Monthly')
df_quarterly = pd.read_excel(file_path, sheet_name='Quarterly')

# 日期
df['yyyymm'] = df['yyyymm'].astype(str)
df['date'] = pd.to_datetime(df['yyyymm'], format='%Y%m') + pd.offsets.MonthEnd(0)
df_quarterly['yyyyq'] = df_quarterly['yyyyq'].astype(str)
df_quarterly['yyyyq'] = df_quarterly['yyyyq'].str[:4] + 'Q' + df_quarterly['yyyyq'].str[4:]
# 1. 统一转换日期（使用 PeriodIndex，简洁且不容易出错）
df_quarterly['date'] = pd.PeriodIndex(df_quarterly['yyyyq'], freq='Q').to_timestamp(how='end')
# 2. 一步到位完成偏移（+4个月 + 月末校准）
# 这里的 DateOffset(months=4) 实际上已经涵盖了你之前逻辑中的 +3个月滞后 +1个月延后
df_quarterly['date'] = df_quarterly['date'] + pd.DateOffset(months=4) + pd.offsets.MonthEnd(0)
# 在所有计算完成后，加这一行即可抹掉时分秒
df_quarterly['date'] = pd.to_datetime(df_quarterly['date']).dt.normalize()

start_date = '1989-01-01'
end_date = '2023-12-31'
df = df[(df['date'] >= start_date) & (df['date'] <= end_date)].reset_index(drop=True)

lag0_cols=['ret','tms','dfy','dfr','svar','vrp','lzrt','wtexas','skvw','tail', 'dtoy', 'dtoat', 'rdsp', 'avgcor']
lag1_cols=['d/p', 'd/y','infl', 'ntis', 'ndrbl', 'disag']
lag4_cols=['e/p', 'd/e', 'ygap','b/m']
quarterly_lag4_cols=['i/k', 'govik', 'crdstd']
# required_cols=['AAA', 'BAA', 'lty', 'ltr', 'tbl', 'd/p', 'd/y', 'e/p', 'd/e', 'b/m', 'ntis', 'svar',  
#                'disag', 'skvw', 'tail', 'rdsp','avgcor','tms', 'dfy', 'dfr', 'infl', 'vrp','ret',]

# 增加新的数据列
required_cols_1996 =['impvar',  'rsvix'] # 从1996年1月才有, 先不加
required_cols_2021 =['vp'] #数据只到2021年末的列, 可以不加, 看情况
# required_cols=required_cols+required_cols_1996+required_cols_2021

# 数值化
# for col in required_cols:
#     # df[col] = pd.to_numeric(df[col], errors='coerce')
#     df[col] = pd.to_numeric(df[col], errors='raise')

# 提取
macro_df_monthly = df_monthly[['date'] + lag0_cols+lag1_cols+lag4_cols].copy()
# 排序
macro_df_monthly = macro_df_monthly.sort_values('date').reset_index(drop=True)

# lag（防未来信息）
for col in lag1_cols:
    if col in macro_df_monthly.columns:
        macro_df_monthly[col] = macro_df_monthly[col].shift(1)
for col in lag4_cols:
    if col in macro_df_monthly.columns:
        macro_df_monthly[col] = macro_df_monthly[col].shift(4)

macro_df_quarterly = df_quarterly[['date'] + quarterly_lag4_cols].copy()
macro_df_quarterly = macro_df_quarterly.sort_values('date').reset_index(drop=True)

# 执行合并
# direction='backward' 意思就是：找之前最近的那个财报日期，并自动填充给后续的月份
macro_df = pd.merge_asof(
    macro_df_monthly, 
    macro_df_quarterly, 
    on='date', 
    direction='backward'
)
macro_df = macro_df[(macro_df['date'] >= '1990-01-01') & (macro_df['date'] <= '2023-12-31')]
macro_df['crdstd'].fillna(54.4, inplace=True)
# # 删除缺失
macro_df.tail(5)

/Users/jianbinchen/miniconda3/envs/research/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/Users/jianbinchen/miniconda3/envs/research/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/var/folders/8b/0krd96r57jldyg4t2s79cv140000gn/T/ipykernel_4327/4268828783.py:67: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].

,date,ret,tms,dfy,dfr,svar,vrp,lzrt,wtexas,skvw,tail,dtoy,dtoat,rdsp,avgcor,d/p,d/y,infl,ntis,ndrbl,disag,e/p,d/e,ygap,b/m,i/k,govik,crdstd
1831,2023-08-31,-0.015424,-0.0113,0.0107,-0.002586,0.001337,2.8534,-1.513901,-0.002078,0.022579,0.525745,0.974495,0.943539,0.037814,0.191842,0.015017,0.015485,0.001908,-0.016677,0.009484,2.632699,0.042479,0.386069,-3.192753,0.214902,0.035180,0.028982,44.8
1832,2023-09-30,-0.047566,-0.0094,0.0103,-0.004600,0.001023,15.3599,-1.589586,0.112214,-0.028320,0.504731,0.940411,0.910539,0.026736,0.195060,0.015333,0.015061,0.004367,-0.016514,0.003226,2.503318,0.042840,0.382809,-3.185364,0.222672,0.035180,0.028982,44.8
1833,2023-10-31,-0.021040,-0.0054,0.0102,-0.006642,0.001678,10.4262,-1.577237,-0.101002,-0.024082,0.484134,0.927652,0.898184,0.028562,0.239102,0.016164,0.015377,0.002485,-0.017989,0.027810,2.510934,0.040673,0.379620,-3.239006,0.212969,0.035653,0.029756,46.0
1834,2023-11-30,0.090804,-0.0077,0.0101,0.025053,0.001341,4.0318,-1.729436,-0.073634,0.048705,0.495785,1.000000,0.976936,0.033699,0.257148,0.016606,0.016241,-0.000383,-0.014711,-0.000194,2.525981,0.039680,0.378463,-3.265166,0.206070,0.035653,0.029756,46.0
1835,2023-12-31,0.045506,-0.0122,0.0090,0.009703,0.000797,5.9600,-1.739012,-0.049200,-0.000504,0.508539,0.999455,0.999455,0.047711,0.239503,0.015319,0.016685,-0.002015,-0.020618,0.063209,2.255945,0.040635,0.377320,-3.243973,0.211041,0.035653,0.029756,46.0


In [38]:
# Save it to Parquet
macro_df.to_parquet('./data/macro_data.parquet', index=False)

这份表单是实证资产定价领域的**绝对宝库**。它不仅包含了 Welch & Goyal (2008) 那 14 个最经典的预测因子，还收录了过去 20 年顶刊新发现的各种宏观异象。

直接按照 1 到 57 顺序讲会非常散乱。作为研究资产定价的同行，我帮你把它们按照**经济学直觉**分成了 6 大模块。这样你在写论文或挑选特征时，思路会极其清晰：

---

### 模块一：市场基础量价与收益 (Market Basics)
这是构建所有其他因子的底层数据，基本都是标普 500 (S&P 500) 的数据。
* **1 date:** 日期。
* **2 price:** 标普 500 指数价格。
* **3 d12 & 4 e12:** 过去 12 个月滚动的总股息 (Dividends) 和总盈利 (Earnings)。
* **5 ret & 6 retx:** 标普 500 的月度收益率（ret 包含分红，retx 不含分红）。
* **13 Rfree:** 无风险收益率（通常用 1 个月期国库券）。

### 模块二：经典估值比率 (Valuation Ratios)
这些是 Campbell-Shiller 时代确立的最正统的预测因子，逻辑是“均值回归”（估值高了未来收益低）。
* **14 d/p (Dividend Price Ratio):** 股息价格比 (d12/price)。
* **15 d/y (Dividend Yield):** 股息率 (d12 / 上期price)。
* **16 e/p (Earnings Price Ratio):** 盈利价格比 (e12/price)。
* **17 d/e (Dividend Payout Ratio):** 股息支付率 (d12/e12)。企业利润中有多少拿来分红。
* **18 b/m (Book-to-Market):** 宏观账面市值比（道琼斯工业平均指数的 B/M）。

### 模块三：利率体系与利差 (Yields & Spreads)
这部分捕捉了宏观货币政策和企业融资环境（信用周期）。
* **7 AAA & 8 BAA:** 最高信用评级 (AAA) 和中等信用评级 (BAA) 的企业债收益率。
* **9 lty & 10 ltr:** 长期国债的收益率 (Yield) 和实际回报率 (Return)。
* **11 corpr:** 长期企业债的实际回报率。
* **12 tbl:** 短期国库券利率 (T-bill)。
* **19 tms (Term Spread):** 期限利差 (lty - tbl)。长短期利率倒挂是衰退的最强前兆。
* **20 dfy (Default Yield Spread):** 违约收益率差 (BAA - AAA)。衡量市场对企业破产的恐慌度。
* **21 dfr (Default Return Spread):** 违约回报率差 (corpr - ltr)。

### 模块四：风险、波动与尾部 (Risk, Volatility & Tails) 🔥（你的研究核心）
这些是你应该**重点关注**的变量，它们与你的 MDN 尾部风险极度相关。
* **25 svar ($\sigma^2$):** 历史股票方差（经典变量）。
* **30 vp, 31 impvar, 32 vrp:** 方差溢价 (Variance Premium)、隐含波动率和方差风险溢价。期权市场定价和实际波动的差值，是预测崩盘的神器。
* **35 skew & 43 skvw:** 偏度 (Skewness) 和股票平均偏度。捕捉市场收益分布的非对称性。
* **44 tail (X-sect tail risk):** 横截面尾部风险！这简直是你 $ES_{5\%}$ 的宏观对标物。
* **49 rdsp (Stock return dispersion):** 股票收益率分散度。横截面上股票各自为战的程度。
* **50 rsvix:** 缩放后的风险中性 VIX 指数。
* **55 avgcor:** 股票收益率的平均相关性。危机时相关性通常会飙升（“所有东西都在跌”）。

### 模块五：宏观经济与企业行为 (Macro & Fundamentals)
反映实体经济冷暖和管理层的资本运作。
* **22 infl:** 通货膨胀率。
* **23 eqis & 24 ntis:** 股权发行比例和净股权发行。管理层是“聪明钱”，疯狂发股票圈钱往往意味着大盘见顶。
* **26 cay:** 极其著名的宏观预测因子（消费、财富、收入的协整残差，Lettau & Ludvigson 2001）。
* **27 i/k & 33 govik:** 投资/资本比率，公共部门投资。
* **29 pce, 51 gpce, 52 gip:** 宏观消费趋势、年末经济增长、工业生产增长。
* **37 ogap:** 产出缺口 (Output gap)。
* **38 wtexas:** 国际油价变动。
* **39 accrul & 40 cfacc:** 财务应计利润项目（盈利质量）。
* **54 house:** 房产/消费比。

### 模块六：行为金融、情绪与微观摩擦 (Behavioral & Frictions)
捕捉投资者的非理性情绪和市场流动性。
* **28 csp:** 横截面溢价。
* **34 lzrt:** 流动性缺失指标 (Illiquidity measures)。
* **36 crdstd:** 银行信贷收紧标准。
* **41 sntm:** Baker & Wurgler 著名的“投资者情绪指数 (Sentiment)”。
* **42 ndrbl:** 耐用品新订单。
* **45 fbm:** 基于 PLS 提取的 B/M 横截面因子。
* **46 dtoy & 47 dtoat:** 距离 52 周最高点、距离历史最高点的距离（锚定效应）。
* **48 ygap:** 股债收益率差 (Fed Model)。
* **53 tchi:** 14 个技术分析指标的汇总。
* **56 shtint:** 做空比例 (Short interest)。
* **57 disag:** 分析师预测分歧度 (Analyst disagreement)。分歧越大，往往未来收益越差。

---

### ⚠️ 顶级学刊排雷指南 (CRITICAL WARNING)

我们在上一轮讨论中之所以只留下 20 个变量，是因为**你的模型是样本外预测 (Out-of-Sample, OOS)。**

请仔细看上面描述中凡是带有 **"Needs recomputation every period. Only full-sample version here"**（比如 `26 cay`, `29 pce`, `37 ogap`, `41 sntm`, `45 fbm`, `53 tchi`, `56 shtint`）的变量。

这表示 Goyal 教授在这个表格里给出的这些列，是**用 2024 年底的所有数据，通过偏最小二乘法 (PLS) 或协整方程向后拟合出来的**。如果你在 2005 年的训练集里输入了 `cay` 或 `sntm`，实际上你是在作弊——你让神经网络“偷看”了 2024 年的未来信息。

**核心结论：**
对于你的 MDN 尾部模型，直接选用**模块一、二、三的经典因子**，配合**模块四 (Risk, Volatility & Tails) 中没有前视偏差的变量（如 vrp, tail, skvw, rdsp）**，是发顶级金融期刊最漂亮、最具防御性的特征矩阵！

# WRDS CRSP Ratios (微观因子)

这是量化基本面研究中最正统、也最纯粹的武器库——**WRDS Financial Ratios Suite (WRDS 官方财务比率库)**。

对于你目前暂时搁置 OAP 异象数据、决定专心用基本面和宏观数据跑通 MDN 基线模型（Baseline）的战略来说，这张表简直完美。它最大的价值在于：WRDS 官方已经帮你把极其复杂的底层财报科目，计算成了**对冲基金最常用的标准化比率**，并且完美对齐了 `public_date`（无前视偏差）。

面对这近 90 个变量，你绝对不能全塞进神经网络（会面临严重的多重共线性和过拟合）。结合你预测 **$ES_{5\%}$ (极左尾崩盘风险)** 的目标，我为你将这些指标分为 **5 大核心战区**，并为你挑出 MDN 最需要的“尾部特征”。

---

### 🚨 战区一：债务与破产雷区 (Solvency & Leverage) - **最核心的左尾触发器**
对于预测极值崩盘，**“公司欠了多少钱，还不还得上”**永远是第一位的。高杠杆公司在宏观环境恶化时，最容易出现断崖式下跌。

* **`short_debt` (短期债务 / 总债务):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 极度致命。短债比例越高，资金链断裂的风险呈指数级上升。
* **`intcov_ratio` (利息保障倍数):** **【MDN 必选 ⭐⭐⭐⭐】** EBIT / 利息支出。衡量公司赚的钱够不够还利息。低于 1 的公司随时暴雷。
* **`debt_at` (总债务 / 总资产):** **【MDN 必选 ⭐⭐⭐】** 最经典的宏观杠杆率。
* **`debt_ebitda` (总债务 / EBITDA):** 华尔街信用评级最看重的指标，比 `debt_at` 更能反映真实的偿债压力。
* *其他同类冗余指标 (建议剔除其一):* `lt_debt`, `debt_capital`, `de_ratio`, `totdebt_invcap`（选 `debt_at` 即可，没必要全放）。

### 🩸 战区二：流动性与现金流 (Liquidity & Cash Flow) - **抗风险的护城河**
当市场恐慌时，利润只是会计数字，**现金才是命**。缺乏流动性的公司在遭遇做空或行业寒冬时毫无抵抗力。

* **`ocf_lct` (经营现金流 / 流动负债):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 这是最真实的短期生存指标。赚不到真金白银还要面临短期债务的公司，左尾极厚。
* **`cash_ratio` (现金比率):** **【MDN 必选 ⭐⭐⭐⭐】** 账面纯现金 / 流动负债。直接衡量“今天如果债主逼债，能不能活下来”。
* **`fcf_ocf` (自由现金流 / 经营现金流):** 反映公司的资本开支压力。
* *其他同类冗余指标:* `curr_ratio` (流动比率), `quick_ratio` (速动比率)。在预测崩盘时，它们不如 `cash_ratio` 狠，因为存货和应收账款在破产时往往一文不值。

### 🎭 战区三：盈余质量 (Accounting Quality) - **财务造假与爆雷预测**
* **`accrual` (应计项目 / 平均资产):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 盈余操纵的试金石！如果一家公司净利润很高，但全在应收账款和存货里（高 accrual），这是未来业绩大爆雷（股价单日腰斩）的最强预兆。

### 💰 战区四：估值与均值回归 (Valuation & Multiples) - **决定下跌的空间**
估值越高，泡沫破裂时的“重力加速度”越大。估值因子是决定收益率分布中心 $\mu$ 和方差 $\sigma$ 的重要锚点。

* **`bm` (账面市值比):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 最经典的价值因子。
* **`pe_exi` (扣非市盈率):** **【MDN 必选 ⭐⭐⭐⭐】** 剔除了非经常性损益的真实市盈率，比 `pe_inc` 更干净。
* **`evm` (企业价值倍数 EV/EBITDA):** 并购和私募股权最爱看的指标，不受资本结构（负债率）影响。
* *其他同类冗余指标:* `pe_op_basic`, `ptb`, `ps` (市销率), `pcf` (市现率)。选 2-3 个核心的即可。

### ⚙️ 战区五：盈利能力与效率 (Profitability & Efficiency)
* **`roe` (净资产收益率):** **【MDN 必选 ⭐⭐⭐⭐】** 基本面质量的终极体现。
* **`gpm` (毛利率) 或 `opmad` (税后营业利润率):** 衡量公司在行业内的护城河深度。毛利被压缩通常是崩盘的早期信号。
* **`at_turn` (总资产周转率):** 运营效率。
* *其他同类冗余指标:* `roa`, `roce`, `aftret_eq` 等高度相关，留 `roe` 足矣。

---

### 🗑️ 垃圾桶：绝对不要放入神经网络的变量

1.  **标识符与日期：** `gvkey`, `adate`, `qdate`, `public_date`（仅用于合并和排序，绝对不能作为 $X$ 输入网络）。
2.  **文本与描述：** `gicdesc`, `ffi48_desc` 等一切 `VARCHAR` 文本列。
3.  **绝对数值：** `mktcap` (总市值), `price` (股价)。**如果你要放，必须对它们进行横截面 Rank 转换**，否则几万亿的市值会瞬间引起梯度爆炸。

---

### 🚀 你的 MDN 基线特征包 (Baseline Feature Set) 建议

为了避免维度灾难，我建议你从这张表里**精挑细选这 15 个黄金比率**作为 $X_{Fundamentals}$：

```python
mdn_features_wrds = [
    # 破产与债务
    'short_debt', 'intcov_ratio', 'debt_at', 'debt_ebitda',
    # 流动性与造假预警
    'ocf_lct', 'cash_ratio', 'accrual', 
    # 估值泡沫
    'bm', 'pe_exi', 'evm', 'divyield',
    # 盈利与效率
    'roe', 'gpm', 'at_turn', 'rd_sale'  # rd_sale (研发占比) 对美股科技股防雷很有用
]
```

将这 15 个特征，加上你处理好的 Goyal 宏观特征（如 `TAIL`, `DEF`, `VIXCLS`），你的第一版输入层 $X_T$ 就能控制在 20 个维度左右。配合上横截面 Rank 标准化，这是一个极其清爽且具备极强经济学解释力的输入矩阵！

这些财务比率中必定存在大量的 `NaN`（比如很多科技公司没有短期债务，或者尚未盈利导致市盈率缺失）。面对这些残缺，在进行 Rank 转换之前，你打算使用全市场的截面中位数填充，还是基于它所在的行业 (`gsector` / `ffi48`) 进行同行业中位数填充？

In [49]:
crsp_wrds_msfv2_query_1990=pd.read_parquet('./data/crsp_wrds_msfv2_query_1990.parquet', engine='pyarrow',columns=['permno',	'mthcaldt',	'mthret',	'mthprc',	'mthcap',	'mthvol','shrout','siccd'])
# crsp_wrds_msfv2_query_1990=pd.read_parquet('./data/crsp_wrds_msfv2_query_1990.parquet', engine='pyarrow')
# 剔除金融行业 (SIC 代码 6000 至 6999)
crsp_wrds_msfv2_query_1990['siccd']=crsp_wrds_msfv2_query_1990['siccd'].astype('Int64')
crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990[(crsp_wrds_msfv2_query_1990['siccd'] < 6000) | (crsp_wrds_msfv2_query_1990['siccd'] > 6999)]
crsp_wrds_msfv2_query_1990.drop(columns=['siccd'], inplace=True)
crsp_wrds_msfv2_query_1990.drop_duplicates(subset=['permno', 'mthcaldt'], keep='first', inplace=True)
crsp_wrds_msfv2_query_1990.head(5)

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout
0,10001,1990-01-31,-0.018519,9.9375,10156.13,35255.0,1022
2,72988,1990-01-31,-0.076923,1.5000,2658.00,13152.0,1772
4,11278,1990-01-31,-0.145161,6.6250,54484.00,325964.0,8224
5,72961,1990-01-31,-0.016640,27.2500,405997.75,1716991.0,14899
8,72741,1990-01-31,-0.013333,74.0000,283124.00,17200.0,3826


In [9]:
crsp_wrds_msfv2_query_1990.shape

(1699533, 7)

In [ ]:
crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990[~crsp_wrds_msfv2_query_1990.duplicated(subset=['permno', 'mthcaldt'], keep='first')]
crsp_wrds_msfv2_query_1990.shape

(2543, 7)

In [47]:
# mdn_features_wrds = [
#     # 破产与债务
#     'short_debt', 'intcov_ratio', 'debt_at', 'debt_ebitda',
#     # 流动性与造假预警
#     'ocf_lct', 'cash_ratio', 'accrual', 
#     # 估值泡沫
#     'bm', 'pe_exi', 'evm', 'divyield',
#     # 盈利与效率
#     'roe', 'gpm', 'at_turn', 'rd_sale',  # rd_sale (研发占比) 对美股科技股防雷很有用
# ]
mdn_features_wrds = [
    # ── 估值（预测均值：价值溢价）───────────────────────────────────────
    'bm',           # 账面市值比，FF 价值因子核心
    'evm',          # EV Multiple，含债务的综合估值
    'pcf',          # Price/Cash Flow，比 PE 更稳健（替换 pe_exi）
    'divyield',     # 股息率

    # ── 盈利质量（预测均值：盈利因子）──────────────────────────────────
    'gprof',        # 毛利润/总资产，Novy-Marx (2013)（替换 roe，信号更干净）
    'npm',          # 净利润率
    'accrual',      # 应计/平均资产，Sloan (1996) 应计异象
    'cfm',          # 现金流利润率（与 accrual 形成对比）

    # ── 成长与投资（预测均值）───────────────────────────────────────────
    'at_turn',      # 资产周转率，FF5 投资因子代理
    'rd_sale',      # R&D/Sales，创新溢价
    'inv_turn',     # 存货周转率

    # ── 杠杆与财务风险（预测方差：尾部风险）────────────────────────────
    'debt_at',      # 总债务/总资产
    'intcov_ratio', # 利息覆盖率，财务困境信号（保留）
    'short_debt',   # 短期债务/总债务，流动性错配（保留）

    # ── 流动性（预测方差）───────────────────────────────────────────────
    'curr_ratio',   # 流动比率（替换 ocf_lct，覆盖率更高）
    'cash_ratio',   # 现金比率（保留）
    # -- 其他--
    'ps', # Price/Sales，对无盈利公司有用
    'opmad', # 经营利润率（折旧后）
    'de_ratio', #财务杠杆
    'mktcap', #取 log → 规模因子
    'capei', #Shiller CAPE，长期均值回归
    'debt_ebitda', # 企业债务/EBITDA，衡量财务压力
    'ocf_lct', # 经营现金流/流动负债，流动性预警
    'gpm', # 毛利率，盈利质量指标
    'roe', # 净资产收益率，盈利能力指标
]
comp_wrds_ratios= pd.read_parquet('./data/comp_wrds_ratios.parquet', engine='pyarrow',columns=['gvkey', 'public_date']+mdn_features_wrds)
comp_wrds_ratios.head(5)

,gvkey,public_date,bm,evm,pcf,divyield,gprof,npm,accrual,cfm,at_turn,rd_sale,inv_turn,debt_at,intcov_ratio,short_debt,curr_ratio,cash_ratio,ps,opmad,de_ratio,mktcap,capei,debt_ebitda,ocf_lct,gpm,roe
0,001003,1990-01-31,NaN,-2.836651,-0.159702,NaN,0.488989,-0.276872,-0.355052,-0.217813,1.554107,0.0,1.565528,0.413638,-4.555412,0.984159,0.960262,0.056425,0.034461,-0.153649,17.424393,0.335375,-0.237956,-1.832834,-0.128283,0.314643,-1.813523
1,013343,1990-01-31,3.597226,6.485974,2.241353,NaN,0.040907,0.299322,-0.044903,0.630680,0.136665,0.0,NaN,0.009276,315.769231,1.000000,NaN,NaN,1.409560,0.299322,0.024770,18.083240,0.386409,0.085077,NaN,0.299322,0.041494
2,002484,1990-01-31,0.715879,NaN,-17.721272,NaN,NaN,0.032242,0.099620,NaN,1.870810,0.0,NaN,0.135434,NaN,0.689936,1.769802,0.201836,0.360000,0.047622,0.900549,248.345907,12.637712,NaN,-0.090208,NaN,0.116036
3,013342,1990-01-31,1.403200,NaN,53.053115,0.046377,0.031130,0.068557,0.023184,0.092006,0.393489,0.0,NaN,0.041482,NaN,0.275118,NaN,NaN,0.532836,0.079114,2.493436,48.808866,0.053272,NaN,NaN,0.079114,0.097703
4,013341,1990-01-31,0.468837,7.854662,1.736912,0.012616,0.071427,0.074888,-0.165359,0.087049,0.556728,0.0,NaN,0.257980,3.942446,0.023124,NaN,NaN,0.629987,0.116919,4.686480,885.824927,13.306532,3.611789,NaN,0.128298,0.247401


In [48]:
comp_wrds_ratios.duplicated(subset=['gvkey', 'public_date'], keep=False).sum()

np.int64(0)

In [50]:
# merge crsp_wrds_msfv2_query_1990 and comp_wrds_ratios by ccm link
with sqlite3.connect(wrds_path) as db_conn:
    crsp_ccmxpf_linktable = pd.read_sql('SELECT * FROM crsp_ccmxpf_linktable', db_conn, parse_dates=['linkdt','linkenddt'])
crsp_ccmxpf_linktable['linkenddt'] = pd.to_datetime(crsp_ccmxpf_linktable['linkenddt']).fillna(pd.to_datetime('2099-12-31'))
crsp_ccmxpf_linktable['lpermno'] = crsp_ccmxpf_linktable['lpermno'].astype("Int64").astype(str).replace("<NA>", pd.NA).str.strip()
crsp_ccmxpf_linktable['gvkey'] = crsp_ccmxpf_linktable['gvkey'].astype(str).str.zfill(6)
crsp_wrds_msfv2_query_1990['permno'] = crsp_wrds_msfv2_query_1990['permno'].astype("Int64").astype(str).replace("<NA>", pd.NA).str.strip()
comp_wrds_ratios['gvkey'] = comp_wrds_ratios['gvkey'].astype(str).str.zfill(6)
ccm = crsp_ccmxpf_linktable[(crsp_ccmxpf_linktable['linktype'].isin(['LU', 'LC'])) & (crsp_ccmxpf_linktable['linkprim'].isin(['P', 'C']))]
ccm = ccm.rename(columns={'lpermno': 'permno'})
comp_ccm = pd.merge(comp_wrds_ratios, ccm[['gvkey', 'permno', 'linkdt', 'linkenddt']], on='gvkey', how='inner')
comp_ccm = comp_ccm[(comp_ccm['public_date'] >= comp_ccm['linkdt']) & 
                    (comp_ccm['public_date'] <= comp_ccm['linkenddt'])]

comp_ccm['public_date'] = comp_ccm['public_date'] + pd.offsets.MonthEnd(0) # 把日期推到月底，和CRSP日期对齐
crsp_wrds_msfv2_query_1990['mthcaldt'] = crsp_wrds_msfv2_query_1990['mthcaldt'] + pd.offsets.MonthEnd(0)# 把日期推到月底
micro_df = pd.merge(
    crsp_wrds_msfv2_query_1990,
    comp_ccm,
    left_on=['permno', 'mthcaldt'],
    right_on=['permno', 'public_date'],
    how='left'
)
micro_df = micro_df.sort_values(['permno', 'public_date']).reset_index(drop=True)

micro_df['mthret_lead1'] = micro_df.groupby('permno')['mthret'].shift(-1)
micro_df['turnover_1m'] = (
    micro_df['mthvol'] /
    micro_df['shrout'].replace(0, np.nan)
)
micro_df['mthcap_log'] = np.log(micro_df['mthcap'].replace(0, np.nan))
# 特征 X4：MOM12m (过去12个月动量，通常跳过最近1个月，即 t-12 到 t-2)
micro_df['log_ret'] = np.log1p(micro_df['mthret'].fillna(0))
# 使用 shift(1) 跳过本月，然后 rolling 11 个月求和
micro_df['MOM12m'] = micro_df.groupby('permno')['log_ret'].transform(
    lambda x: x.shift(1).rolling(window=11, min_periods=8).sum()
)
micro_df['MOM12m'] = np.exp(micro_df['MOM12m']) - 1
micro_df.drop(columns=[ 'public_date', 'gvkey', 'linkdt', 'linkenddt', 'log_ret'], errors='ignore', inplace=True)
micro_df.to_parquet('./data/micro_data.parquet', index=False)
micro_df.sample(5)


/Users/jianbinchen/miniconda3/envs/research/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,bm,evm,pcf,divyield,gprof,npm,accrual,cfm,at_turn,rd_sale,inv_turn,debt_at,intcov_ratio,short_debt,curr_ratio,cash_ratio,ps,opmad,de_ratio,mktcap,capei,debt_ebitda,ocf_lct,gpm,roe,mthret_lead1,turnover_1m,mthcap_log,MOM12m
1607960,90855,2009-03-31,0.000000,0.24000,9982.08,3435508.0,41592,NaN,-1.190198,-0.431475,NaN,0.085461,-95.348548,0.095284,-94.439834,0.021204,65.182573,NaN,4.229787,-42.937262,0.000671,0.706263,0.549676,43.079502,-93.713693,-1.227824,10.382160,-0.471167,-0.533309,-8.661627,1.000000,NaN,1.000000,82.600212,9.208547,-0.733333
228181,15457,2002-11-30,0.043902,6.42000,36831.54,175307.0,5737,0.770951,10.336184,3.879998,NaN,0.210767,0.114112,-0.106386,0.148069,0.655057,0.000000,3.725534,0.217156,17.384853,0.066579,3.219288,1.026264,1.068264,0.236945,0.717192,50.711580,10.007416,1.859946,1.279648,0.321754,0.130985,0.073209,30.557260,10.514110,-0.542751
240666,15950,2018-08-31,0.084677,2.69000,98714.93,2964419.0,36697,0.551126,-2.604779,-1.988155,NaN,-0.666214,-10.867689,-0.056917,-10.627067,0.062595,9.314240,NaN,0.095998,NaN,0.088114,6.461726,6.245234,19.877136,-11.010488,0.417484,98.550840,-2.325249,-0.144095,-5.101660,-10.643203,-0.987130,-0.104089,80.780963,11.499991,0.142856
1642762,91661,2011-04-30,-0.070588,1.58000,73569.54,3779117.0,46563,1.660129,9.575467,-7.261107,NaN,0.085002,0.029458,0.077760,0.100726,0.746018,0.000000,3.989071,0.212526,3.281297,0.655631,2.371804,0.375010,0.525260,0.046972,0.464032,73.569540,11.310174,3.202862,-0.221383,0.113942,0.032712,-0.253165,81.161373,11.205986,-0.174758
872921,76614,2000-12-31,0.446795,35.26563,1205978.75,6797532.0,34197,0.159263,-73.721964,-75.556001,NaN,-0.071416,-0.280214,0.015830,-0.206792,0.302722,1.028004,9.777913,0.021086,-61.552632,0.326435,8.168555,6.935911,24.950558,-0.314696,0.181076,1298.127656,-65.084878,-0.295258,-1.333463,-0.235911,-0.116254,-0.071334,198.775682,14.002802,0.911762


In [52]:
micro_df.head(5)

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,bm,evm,pcf,divyield,gprof,npm,accrual,cfm,at_turn,rd_sale,inv_turn,debt_at,intcov_ratio,short_debt,curr_ratio,cash_ratio,ps,opmad,de_ratio,mktcap,capei,debt_ebitda,ocf_lct,gpm,roe,mthret_lead1,turnover_1m,mthcap_log,MOM12m
0,10001,1990-01-31,-0.018519,9.9375,10156.13,35255.0,1022,0.917331,4.853678,7.769246,0.050633,0.164538,0.050236,-0.005309,0.081263,1.223659,0.0,91.308370,0.426239,3.131479,0.107954,1.173138,0.271814,0.421441,0.103437,2.235379,10.092250,16.372132,2.590528,0.339098,0.134464,0.151031,-0.006289,34.496086,9.225833,NaN
1,10001,1990-02-28,-0.006289,9.8750,10092.25,14862.0,1022,0.849363,4.626443,5.646409,0.050000,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.403697,0.107955,2.084969,10.220000,15.386932,2.279116,0.508677,0.139398,0.162685,0.012821,14.542074,9.219523,NaN
2,10001,1990-03-31,0.012821,9.8750,10141.63,12727.0,1027,0.849363,4.626443,5.744959,0.049383,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.410743,0.107955,2.084969,10.398375,15.655488,2.279116,0.508677,0.139398,0.162685,0.000000,12.392405,9.224404,NaN
3,10001,1990-04-30,0.000000,9.8750,10141.63,16645.0,1027,0.849363,4.626443,5.674033,0.050000,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.405672,0.107955,2.084969,10.270000,15.462210,2.279116,0.508677,0.139398,0.162685,-0.012658,16.207400,9.224404,NaN
4,10001,1990-05-31,-0.012658,9.7500,10013.25,27885.0,1027,0.911008,4.819588,4.424768,0.051282,0.182316,0.056644,-0.047016,0.089212,1.223747,0.0,89.841695,0.403169,3.227166,0.066927,1.278309,0.263992,0.422964,0.116415,1.943177,10.013250,14.173036,2.211369,0.693056,0.148982,0.158487,0.013750,27.151899,9.211664,NaN


In [51]:
micro_df.shape

(1699533, 36)

In [53]:
micro_df.duplicated(subset=['permno', 'mthcaldt'], keep=False).sum()

np.int64(0)

In [51]:
micro_df.tail(5)

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,short_debt,intcov_ratio,debt_at,debt_ebitda,ocf_lct,cash_ratio,accrual,bm,pe_exi,evm,divyield,roe,gpm,at_turn,rd_sale,mthret_lead1,turnover_1m,mthcap_log,MOM12m
1702071,93436,2025-08-31,0.083044,333.87,1.076881e+09,1.598393e+09,3225449,0.234716,15.830137,0.106271,1.144757,0.529209,1.224750,-0.081217,0.075492,199.922156,88.014082,NaN,0.082826,0.236529,0.748345,0.057302,0.332015,591.736144,20.797334,0.439773
1702072,93436,2025-09-30,0.332015,444.72,1.478249e+09,1.966931e+09,3324000,0.234716,15.830137,0.106271,1.144757,0.529209,1.224750,-0.081217,0.075492,266.299401,88.014082,NaN,0.082826,0.236529,0.748345,0.057302,0.026624,608.674801,21.114124,0.276115
1702073,93436,2025-10-31,0.026624,456.56,1.518436e+09,2.024342e+09,3325819,0.234716,15.830137,0.106271,1.144757,0.529209,1.224750,-0.081217,0.075492,273.389222,88.014082,NaN,0.082826,0.236529,0.748345,0.057302,-0.057802,484.040838,21.140947,0.779946
1702074,93436,2025-11-30,-0.057802,430.17,1.430668e+09,1.609832e+09,3325819,0.225793,13.948424,0.105348,1.234543,0.525494,1.285071,-0.084740,0.054098,298.729167,107.865207,NaN,0.068929,0.232817,0.750824,0.061736,0.045447,486.593451,21.081407,0.322748
1702075,93436,2025-12-31,0.045447,449.72,1.495687e+09,1.618322e+09,3325819,0.225793,13.948424,0.105348,1.234543,0.525494,1.285071,-0.084740,0.054098,312.305556,107.865207,NaN,0.068929,0.232817,0.750824,0.061736,NaN,NaN,21.125852,0.065198
